In [ ]:
import os
from getpass import getpass
from pathlib import Path
from langchain_community.document_loaders import (
    PyMuPDFLoader,
    TextLoader,
    Docx2txtLoader,
    DirectoryLoader
)
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings

In [ ]:
DATA_DIR = Path("data")

In [ ]:
loaders = [
    DirectoryLoader(DATA_DIR, glob="**/*.pdf", loader_cls=PyMuPDFLoader, show_progress=True),
    DirectoryLoader(DATA_DIR, glob="**/*.txt", loader_cls=TextLoader, show_progress=True),
    DirectoryLoader(DATA_DIR, glob="**/*.docx", loader_cls=Docx2txtLoader, show_progress=True),
]

docs = []
for loader in loaders:
    docs.extend(loader.load())

In [ ]:
if not os.getenv("OPENAI_API_KEY_EMBEDDINGS"):
    os.environ["OPENAI_API_KEY_EMBEDDINGS"] = getpass("Enter your RCP API key for embedding model: ")

embeddings = OpenAIEmbeddings(
    model="Qwen/Qwen3-Embedding-8B",
    base_url="https://inference.rcp.epfl.ch/v1",
    api_key=os.getenv("OPENAI_API_KEY_EMBEDDINGS")
)

In [ ]:
qdrant = QdrantVectorStore.from_documents(
    docs,
    embeddings,
    url="http://localhost:6333",
    collection_name="basic_split_pages_all_formats",
)